# Relation KG Experiment (Unified)

단일 정본 관계그래프(`relation_edges_confirmed.csv`) 기준 D 실험 노트북입니다.


In [ ]:
# Optional installs
# %pip install -q openai pandas python-dotenv tqdm


In [ ]:
import os
import subprocess
from pathlib import Path
import pandas as pd

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

BASE = Path('/Users/yiji/Desktop/work/pseudo_lab/game_translation_exp')
SCRIPTS = BASE / 'scripts'
DATA = BASE / 'data'
REL = DATA / 'relation_kg'
OUT = BASE / 'outputs'
EVAL = BASE / 'eval'

if load_dotenv is not None:
    load_dotenv(BASE / '.env', override=False)
    load_dotenv(override=False)

RUN_DATE = os.getenv('RUN_DATE', '2026-04-12')
DRY_RUN = os.getenv('DRY_RUN', 'true').lower() == 'true'

print('RUN_DATE =', RUN_DATE)
print('DRY_RUN =', DRY_RUN)
print('OPENAI_API_KEY loaded =', bool(os.getenv('OPENAI_API_KEY')))


In [ ]:
# 1) Optional: refresh auto candidates (not canonical)
REFRESH_AUTO = False
if REFRESH_AUTO:
    cmd = ['python3', str(SCRIPTS / 'extract_relation_candidates.py'), '--allow-cooccurrence']
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)


In [ ]:
# 2) Build sample relation context from canonical confirmed edges
cmd = [
    'python3', str(SCRIPTS / 'build_relation_context.py'),
    '--samples-csv', str(DATA / 'samples_tagged_v1.csv'),
    '--edges-csv', str(REL / 'relation_edges_confirmed.csv'),
    '--seed-csv', str(REL / 'character_seeds.csv'),
    '--out-csv', str(REL / 'sample_relation_context.csv')
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

ctx = pd.read_csv(REL / 'sample_relation_context.csv')
print('context rows:', len(ctx))
ctx.head(10)


In [ ]:
# 3) Inspect rows with explicit relation context
ctx = pd.read_csv(REL / 'sample_relation_context.csv')
has_ctx = ctx[~ctx['relation_context'].str.contains('No explicit relation context', na=False)]
print('with context:', len(has_ctx), '/', len(ctx))
has_ctx[['sample_id','sentence_type','source_text','detected_entities','relation_context']].head(20)


In [ ]:
# 4) Run D condition
cmd = ['python3', str(SCRIPTS / 'run_condition_d.py'), '--run-date', RUN_DATE]
if DRY_RUN:
    cmd.append('--dry-run')
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

df_d = pd.read_csv(OUT / f'run_{RUN_DATE}' / 'D_outputs.csv')
print('D outputs:', len(df_d))
df_d.head(10)


In [ ]:
# 5) Build A/B/C/D comparison sheet
run_dir = OUT / f'run_{RUN_DATE}'
samples = pd.read_csv(DATA / 'samples.csv')

A = pd.read_csv(run_dir / 'A_outputs.csv') if (run_dir / 'A_outputs.csv').exists() else None
B = pd.read_csv(run_dir / 'B_outputs.csv') if (run_dir / 'B_outputs.csv').exists() else None
C = pd.read_csv(run_dir / 'C_outputs.csv') if (run_dir / 'C_outputs.csv').exists() else None
D = pd.read_csv(run_dir / 'D_outputs.csv') if (run_dir / 'D_outputs.csv').exists() else None

merged = samples[['sample_id','sentence_type','source_text']].copy()
if A is not None:
    merged = merged.merge(A[['sample_id','translation']].rename(columns={'translation':'condition_A'}), on='sample_id', how='left')
if B is not None:
    merged = merged.merge(B[['sample_id','translation']].rename(columns={'translation':'condition_B'}), on='sample_id', how='left')
if C is not None:
    merged = merged.merge(C[['sample_id','translation']].rename(columns={'translation':'condition_C'}), on='sample_id', how='left')
if D is not None:
    merged = merged.merge(D[['sample_id','translation']].rename(columns={'translation':'condition_D'}), on='sample_id', how='left')

for col in [
    'A_meaning','A_term','A_consistency','A_naturalness','A_ui_fit',
    'B_meaning','B_term','B_consistency','B_naturalness','B_ui_fit',
    'C_meaning','C_term','C_consistency','C_naturalness','C_ui_fit',
    'D_meaning','D_term','D_consistency','D_naturalness','D_ui_fit',
    'best','error_type','notes'
]:
    if col not in merged.columns:
        merged[col] = ''

out_eval = EVAL / f'eval_sheet_prefilled_ABCD_{RUN_DATE}.csv'
merged.to_csv(out_eval, index=False, encoding='utf-8')
print('saved:', out_eval)
print('rows:', len(merged))


## Notes
- Canonical source: `data/relation_kg/relation_edges_confirmed.csv`
- `.env`는 `game_translation_exp/.env`와 현재 작업 디렉터리에서 로딩합니다.
